# 2.5 Colonisation forecast updater (**Colonisation forecast**)
Fits logistic / Gompertz / Avrami curves to the coverage table and tests the forecast. The curve code now lives in `mycodemo/growth/forecast.py` (shared with the app). Run notebook 2.4 first (it writes data/interim/coverage_auto.csv).

### Cell 1. Load and define the three curve shapes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from mycodemo.config import INTERIM_DATA_DIR
from mycodemo.growth.forecast import MODELS, avrami, day_reaching, gompertz, logistic

cov = pd.read_csv(INTERIM_DATA_DIR / "coverage_auto.csv")

### Cell 2. Fit one dish and plot** (the "hello world")

In [ ]:
g = cov[cov.series_id == "d005-H"].sort_values("day")
t, c = g["day"].values, g["coverage"].values
tt = np.linspace(0, t.max() + 3, 200)
plt.plot(t, c, "ko", label="photos")
for name, (f, p0) in MODELS.items():
    params, _ = curve_fit(f, t, c, p0=p0, maxfev=10000)
    plt.plot(tt, f(tt, *params), label=name)
plt.xlabel("day"); plt.ylabel("coverage"); plt.legend(); plt.show()

### Cell 3. Day of 90% / 95% from a fitted curve

In [ ]:
# day_reaching(f, params, p) is imported from mycodemo.growth.forecast in cell 1
day_reaching

### Cell 4. Which shape fits best? (step 3 of the plan)

In [ ]:
results = []
for sid, g in cov.groupby("series_id"):
    t, c = g["day"].values, g["coverage"].values
    for name, (f, p0) in MODELS.items():
        try:
            params, _ = curve_fit(f, t, c, p0=p0, maxfev=10000)
            rmse = np.sqrt(np.mean((f(t, *params) - c) ** 2))
        except RuntimeError:
            rmse = np.nan
        results.append({"series_id": sid, "model": name, "rmse": rmse})
fit = pd.DataFrame(results).pivot(index="series_id", columns="model", values="rmse")
fit

### Cell 5. The forecasting test (step 4 of the plan)

In [ ]:
rows = []
for sid, g in cov.groupby("series_id"):
    g = g.sort_values("day")
    t, c = g["day"].values, g["coverage"].values
    # "truth": day 90% from the full series
    p_full, _ = curve_fit(logistic, t, c, p0=[1, 5], maxfev=10000)
    true90 = day_reaching(logistic, p_full, 0.90)
    # baseline: average 90% day of the OTHER dishes (filled in below)
    for n_photos in range(3, len(t)):
        try:
            p_part, _ = curve_fit(logistic, t[:n_photos], c[:n_photos], p0=[1, 5], maxfev=10000)
            pred = day_reaching(logistic, p_part, 0.90)
        except RuntimeError:
            pred = np.nan
        rows.append({"series_id": sid, "n_photos": n_photos, "pred90": pred, "true90": true90})
fc = pd.DataFrame(rows)
true_by_dish = fc.groupby("series_id")["true90"].first()
fc["baseline90"] = [true_by_dish.drop(s).mean() for s in fc["series_id"]]
fc["err_model"] = (fc["pred90"] - fc["true90"]).abs()
fc["err_baseline"] = (fc["baseline90"] - fc["true90"]).abs()
summary = fc.groupby("n_photos")[["err_model", "err_baseline"]].median()
summary.plot(marker="o"); plt.ylabel("error in days (median)"); plt.show()
summary

### Cell 6. Save the training history
Learning curve of the **Colonisation forecast**: the logistic curve is fitted to the first `train_size` photos of each dish. `train_score` is the fit error (RMSE of coverage) on those photos, `val_score` the error on the same dish's later photos (the forecast is about the future of one growth, so the split is in time). `err90_model_days` and `err90_baseline_days` are the median error of the 90% day against the average of the other dishes (leave one dish out), from the table above. CSV and PNG go to `Desktop/MycoMaterials/kaggle/history/`; plot several runs with `plot_history.plot_runs`.

In [ ]:
import datetime
from mycodemo.config import HISTORY_DIR
from mycodemo.plot_history import save_history

rows = []
for sid, g in cov.groupby("series_id"):
    g = g.sort_values("day")
    t, c = g["day"].values, g["coverage"].values
    for n_photos in range(3, len(t)):
        try:
            p_part, _ = curve_fit(logistic, t[:n_photos], c[:n_photos], p0=[1, 5], maxfev=10000)
        except RuntimeError:
            continue
        rmse = lambda x, y: float(np.sqrt(np.mean((logistic(x, *p_part) - y) ** 2)))
        rows.append({"train_size": n_photos, "train_score": rmse(t[:n_photos], c[:n_photos]),
                     "val_score": rmse(t[n_photos:], c[n_photos:])})
hist = (pd.DataFrame(rows).groupby("train_size").agg(train_score=("train_score", "mean"),
        val_score=("val_score", "mean"), n_dishes=("val_score", "size")).reset_index())
hist = hist.merge(summary.rename(columns={"err_model": "err90_model_days", "err_baseline": "err90_baseline_days"})
                  .reset_index().rename(columns={"n_photos": "train_size"}), on="train_size", how="left")
csv, png = save_history(
    hist, HISTORY_DIR, "d005dishes", model="colonisation_forecast", run="run1",
    notebook="2.5.colonisation_forecast_updater", train_set="first n photos of each d005 dish",
    test_set="later photos of the same dish", date=datetime.date.today().isoformat(),
    settings={"curve": "logistic", "score": "RMSE of coverage (lower is better)"})
print(csv, png, sep="\n")
hist